In [7]:
# import storage account helper
import sys
from pathlib import Path
from dotenv import load_dotenv
import os
project_root = Path().resolve().parent  # from neat_dashboard/ to repo root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

ml_monolith_root = project_root / "ml_monolith"
if str(ml_monolith_root) not in sys.path:
    sys.path.insert(0, str(ml_monolith_root))

from ml_monolith.data_pipeline.storage_account_helpers import download_blob_to_dir, list_blobs_in_prefix
from ml_monolith.data_pipeline.process_raw_data import merge_json_files, label_data, label_data_v2, SensorRecording
load_dotenv()

storage_account_blob_uri = 'wasbs://validation-data@harmlstorage.blob.core.windows.net'
items_list = list_blobs_in_prefix(storage_account_blob_uri)

print("items in blob storage:")
for folder in items_list:
    print(folder)


2026-03-03 17:37:10,234 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Using DefaultAzureCredential for authentication.
2026-03-03 17:37:10,235 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-03-03 17:37:10,235 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2026-03-03 17:37:10,236 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Listing blobs with prefix: 
2026-03-03 17:37:10,237 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.12.3 (Linux-6.17.9-76061709-generic-x86_64-with-glibc2.39)'
No body was attached to the request
2026-03-03 17:37:11,567 - azure.identity._credentials.chained - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
202

items in blob storage:
1772543856762.json
1772543856762_seg0.json
1772544018347.json
1772544018347_seg0.json
1772544209074.json
1772544373575.json
1772544529229.json
1772544674631.json
1772549822805.json
1772552307399.json
1772552307399_seg0.json
labels.csv


In [ ]:
download_dir = "validation/downloaded_data/03-03-2026"

download_blob_to_dir(storage_account_blob_uri, download_dir)

2026-03-03 17:37:16,589 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Using DefaultAzureCredential for authentication.
2026-03-03 17:37:16,590 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-03-03 17:37:16,591 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2026-03-03 17:37:16,591 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Listing blobs with prefix: 
2026-03-03 17:37:16,592 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.12.3 (Linux-6.17.9-76061709-generic-x86_64-with-glibc2.39)'
No body was attached to the request
2026-03-03 17:37:17,947 - azure.identity._credentials.chained - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
202

In [ ]:
download_dir = "validation/downloaded_data"

folders = sorted(os.listdir(download_dir))

from datetime import datetime

def parse_date_prefix(name: str) -> datetime:
    date_part = name.split("_", 1)[0]
    return datetime.strptime(date_part, "%d-%m-%Y")

sorted_folders = sorted(folders, key=parse_date_prefix)
sorted_folders

['03-03-2026']

In [ ]:
output_dir = "validation/labeled_data"

for folder in sorted_folders:
    folder_to_process = os.path.join(download_dir, folder)
    merged_data = merge_json_files(folder_to_process, skipped_files=[])
    csv_files = [f for f in os.listdir(folder_to_process) if f.lower().endswith(".csv")]
    labels_csv_path = os.path.join(folder_to_process, csv_files[0]) if csv_files else None
    if folder in ['15-01-2026', '26-01-2026']:
        print(f"Folder {folder} needs to be processed with label_data.")
        labeled_data_df = label_data(merged_data, labels_csv_path)
    else:
        print(f"Folder {folder} needs to processed with label_data_v2.")
        labeled_data_df = label_data_v2(merged_data, labels_csv_path)
    
    output_path = os.path.join(output_dir, f"{folder}_labeled.csv")
    os.makedirs(output_dir, exist_ok=True)
    labeled_data_df.to_csv(output_path, index=False)


2026-03-03 17:37:27,737 - ml_monolith.data_pipeline.process_raw_data - INFO - Validating directory: neat_dashboard/downloaded_data/03-03-2026
2026-03-03 17:37:27,737 - ml_monolith.data_pipeline.process_raw_data - INFO - Directory neat_dashboard/downloaded_data/03-03-2026 exists.
2026-03-03 17:37:27,738 - ml_monolith.data_pipeline.process_raw_data - INFO - Path neat_dashboard/downloaded_data/03-03-2026 is a directory.
2026-03-03 17:37:27,738 - ml_monolith.data_pipeline.process_raw_data - INFO - Directory neat_dashboard/downloaded_data/03-03-2026 is not empty.
2026-03-03 17:37:27,739 - ml_monolith.data_pipeline.process_raw_data - ERROR - The file labels.csv is not a JSON file.
2026-03-03 17:37:27,739 - ml_monolith.data_pipeline.process_raw_data - INFO - File extension: .csv
2026-03-03 17:37:27,802 - ml_monolith.data_pipeline.process_raw_data - INFO - All files in directory neat_dashboard/downloaded_data/03-03-2026 are valid.
2026-03-03 17:37:27,802 - ml_monolith.data_pipeline.process_raw

Folder 03-03-2026 needs to processed with label_data_v2.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13461 entries, 0 to 13460
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   accelerometerX  13461 non-null  float64
 1   accelerometerY  13461 non-null  float64
 2   accelerometerZ  13461 non-null  float64
 3   gyroscopeX      13461 non-null  float64
 4   gyroscopeY      13461 non-null  float64
 5   gyroscopeZ      13461 non-null  float64
 6   timestamp       13461 non-null  int64  
 7   timestampNanos  13461 non-null  int64  
 8   label           10099 non-null  object 
dtypes: float64(6), int64(2), object(1)
memory usage: 946.6+ KB


In [ ]:
import pandas as pd
generation_seed_dir = "validation/generation_seed"
os.makedirs(generation_seed_dir, exist_ok=True)


def merge_csv_files(csv_dir):
    merged_df = pd.DataFrame()
    files_sorted = sorted(os.listdir(csv_dir))
    for filename in files_sorted:
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(csv_dir, filename))
            merged_df = pd.concat([merged_df, df], ignore_index=True)
    return merged_df


merged_csv_df = merge_csv_files(output_dir)
merged_csv_path = os.path.join(generation_seed_dir, "validation_set.csv")
merged_csv_df.to_csv(merged_csv_path, index=False)

